In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from absl import app, flags, logging
from ml_collections import config_flags

import sys
import os

sys.path.append(os.path.abspath(".."))

import linear.train_mixture as tmix
from linear.mixture_task import MixtureOfGaussiansRegression, DiscreteInputLinearRegression
from linear.linear_utils import *
from tqdm.notebook import trange

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering

import matplotlib.patches as patches
from scipy.optimize import curve_fit

import plotly.io as pio
pio.renderers.default = "notebook_connected"

import plotly.graph_objects as go

logging.set_verbosity(logging.INFO)
torch.set_printoptions(precision=3, sci_mode=False)
np.set_printoptions(precision=3, suppress=True)

%load_ext autoreload
%autoreload 2

### Mixture

In [5]:
config = tmix.get_mixture_config()
print("Training on Mixture of Gaussians Regression task...")

model, log = tmix.train(config, verbose=False)

Training on Mixture of Gaussians Regression task...
train_f84dd5b91dc37b94cb1794ac10f08561 already completed
Loaded model from results/mixture_of_gaussians_regression/train_f84dd5b91dc37b94cb1794ac10f08561/checkpoint.pt


In [12]:
import json
from ml_collections import ConfigDict
import os
from attn_plots_beta import visualize_attention

work_dir = os.path.join("results", "mixture_of_gaussians_regression")
exp_dir = os.path.join(work_dir, "train_f84dd5b91dc37b94cb1794ac10f08561")
config_path = os.path.join(exp_dir, "config.json")
with open(config_path, "r") as f:
    config_dict = json.load(f)

config = ConfigDict(config_dict)
checkpoint_path = os.path.join(exp_dir, "checkpoint.pt")
checkpoint = torch.load(checkpoint_path, map_location=config.device)
data_type = torch.float
model = get_model(**config["model"], dtype=data_type)
model.load_state_dict(checkpoint["model"])
model = model.to(config.device)
train_task = tmix.get_task(**config["task"])
train_task.batch_size = 1
demo_data, demo_component_labels, demo_target = train_task.sample_from_task(train_task.task_pool1[0], train_task.task_pool2[0], step=1)
attns = get_attn(model, demo_data, demo_target)
cap = 100
attns_capped = {layer_key: tensor[:, :cap, :cap] for layer_key, tensor in attns.items()}

widget = visualize_attention(attns_capped, mode='widget')
widget

In [19]:
train_task = tmix.get_task(**config["task"])
train_task.batch_size = 1024
l0 = 2
task_vectors = compute_task_vectors(config, model, train_task, l0)

In [20]:
plot_task_vector_variance_with_fit(task_vectors)

In [21]:
plot_task_vector_differences(task_vectors)

In [22]:
plot_pairwise_task_vector_variance(task_vectors)

In [24]:
tvs_means = task_vectors.mean(dim=-2)
tvs_mean_weighted = tvs_means[:, -10:].mean(dim=1)# weighted_average_favor_late_batched(tvs_means, mode="exp")

t0 = 10
train_task.batch_size = 256
print(f"Memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

for k in range(config.task.n_tasks):
    torch.cuda.empty_cache()
    task_idx = k
    query_data, _, query_target = train_task.sample_from_task(train_task.task_pool1[task_idx], train_task.task_pool2[task_idx], step=600)
    preds_rand = predict_with_task_vector(
            model=model,
            query_data=query_data[:, :(3*t0+3)],
            query_target=query_target[:, :(3*t0+3)],
            task_vector=tvs_mean_weighted[task_idx],
            l=l0,              # same layer
            pad="mapsto",      # same position
            pos=3*t0+1
        )
    with torch.no_grad():
        preds = model(query_data, query_target)
    loss = nn.MSELoss()(preds_rand[:,t0], query_target[:,t0].to(preds_rand.device))
    baseline_loss = nn.MSELoss()(preds[:,t0], query_target[:,t0].to(preds_rand.device))
    print(f"{t0}-shot loss w. injected task vector: {loss.item():.6f}")
    print(f"{t0}-shot loss w.o. injected task vector: {baseline_loss.item():.6f}")

Memory allocated: 0.03 GB
10-shot loss w. injected task vector: 10.609373
10-shot loss w.o. injected task vector: 0.252368
10-shot loss w. injected task vector: 5.005216
10-shot loss w.o. injected task vector: 0.250858
10-shot loss w. injected task vector: 3.133928
10-shot loss w.o. injected task vector: 0.248418


### Discrete

In [27]:
config = tmix.get_discrete_config()
print("Training on Discrete Input Regression task...")

model, log = tmix.train(config, verbose=False)

Training on Discrete Input Regression task...
train_0830233fefcd11d6c629f5b365b7a977 already completed
Loaded model from results/discrete_input_regression/train_0830233fefcd11d6c629f5b365b7a977/checkpoint.pt


In [28]:
import json
from ml_collections import ConfigDict
import os
from attn_plots_beta import visualize_attention

work_dir = os.path.join("results", "discrete_input_regression")
exp_dir = os.path.join(work_dir, "train_0830233fefcd11d6c629f5b365b7a977")
config_path = os.path.join(exp_dir, "config.json")
with open(config_path, "r") as f:
    config_dict = json.load(f)

config = ConfigDict(config_dict)
checkpoint_path = os.path.join(exp_dir, "checkpoint.pt")
checkpoint = torch.load(checkpoint_path, map_location=config.device)
data_type = torch.float
model = get_model(**config["model"], dtype=data_type)
model.load_state_dict(checkpoint["model"])
model = model.to(config.device)
train_task = tmix.get_task(**config["task"])
train_task.batch_size = 1
demo_data, demo_target = train_task.sample_from_task(train_task.task_pool[0], step=1)
attns = get_attn(model, demo_data, demo_target)
cap = 100
attns_capped = {layer_key: tensor[:, :cap, :cap] for layer_key, tensor in attns.items()}

widget = visualize_attention(attns_capped, mode='widget')
widget

In [33]:
train_task = tmix.get_task(**config["task"])
train_task.batch_size = 1024
l0 = 1
task_vectors = compute_task_vectors(config, model, train_task, l0)

In [34]:
plot_task_vector_variance_with_fit(task_vectors)

In [35]:
plot_task_vector_differences(task_vectors)

In [36]:
plot_pairwise_task_vector_variance(task_vectors)